# Project 4 Extension — CNN-GRU for QuitoBench Time-Series Forecasting

## Introduction

This notebook extends Project 4 (GRU-based text generation) to a **multivariate time-series
forecasting** task. We implement and compare two architectures:

| Model | Architecture |
|---|---|
| **Plain GRU** | GRU → GRU → Dense (baseline) |
| **CNN-GRU** | Conv1D → Dropout → Conv1D → Dropout → GRU → GRU → Dense |

The CNN-GRU architecture follows Sajjad et al., *A Novel CNN-GRU-Based Hybrid Approach for
Short-Term Residential Load Forecasting*, IEEE Access 2020. CNN layers act as feature
extractors over local time windows before the stacked GRUs model long-range temporal
dependencies.

### Dataset: QuitoBench

[QuitoBench](https://huggingface.co/datasets/hq-bench/quitobench) is a regime-balanced
evaluation benchmark curated from **Quito**, a billion-scale, single-provenance time-series
dataset of application-traffic workloads from Alipay's production platform.

* Resolution: **1 minute** (`min` split)
* Each row: one timestamp of one series (long / tidy format)
* Five anonymised traffic variates **`ind_1` ... `ind_5`** (float64, NaN for missing)

**Task**: given a window of the past **T = 24** minutes, predict all five variates at the next
minute (one-step-ahead multivariate forecasting).

### Reuse from Project 4

* `layers.Layer`, `layers.Dense`, `layers.Dropout` — base layer infrastructure
* `rnn_layers.GRU` — custom GRU implementation (statically unrolled, `@tf.function`-compatible)
* `network.DeepNetwork` — `early_stopping`, `lr_step_decay`, `save_wts`/`load_wts`
* **New in this extension**: `conv1d_layer.Conv1D`, `time_series_net.{GRU_Regressor, CNN_GRU_Regressor}`


---
## 1. Setup

In [ ]:
%pip install -q datasets pandas scikit-learn

In [ ]:
import sys, os

PROJECT4_DIR  = os.path.abspath('..')
EXTENSION_DIR = os.path.abspath('.')
for p in [PROJECT4_DIR, EXTENSION_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from conv1d_layer import Conv1D
from time_series_net import GRU_Regressor, CNN_GRU_Regressor

print('TensorFlow:', tf.__version__)
print('NumPy:     ', np.__version__)


---
## 2. Data Loading & Preprocessing

In [ ]:
from datasets import load_dataset

print('Loading QuitoBench (minute resolution) ...')
ds_min  = load_dataset('hq-bench/quitobench', 'min')
df_full = ds_min['test'].to_pandas()

print(f'Full dataset shape  : {df_full.shape}')
print(f'Unique series (item_id): {df_full["item_id"].nunique()}')
df_full.head()


In [ ]:
# Pick the series with the most data points
counts    = df_full['item_id'].value_counts()
chosen_id = counts.index[0]
print(f'Chosen item_id: {chosen_id}  ({counts[chosen_id]:,} rows)')

series = (df_full[df_full['item_id'] == chosen_id]
          .sort_values('date_time').reset_index(drop=True))

FEATURE_COLS = ['ind_1', 'ind_2', 'ind_3', 'ind_4', 'ind_5']
N_FEATURES   = len(FEATURE_COLS)

feat_df = series[FEATURE_COLS].copy()
print(f'Raw shape      : {feat_df.shape}')
print(f'NaN per column :\n{feat_df.isna().sum().to_string()}')


In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 1. Impute missing values (forward-fill then backward-fill)
feat_df = feat_df.ffill().bfill().fillna(0.0)
print(f'NaN after imputation: {feat_df.isna().sum().sum()}')

# 2. MinMax-normalise each feature to [0, 1]
scaler          = MinMaxScaler()
features_scaled = scaler.fit_transform(feat_df.values).astype('float32')

# 3. Build sliding-window dataset: X=(B,T,F), y=(B,F)
WINDOW = 24   # 24 minutes of history -> predict the next minute

def make_windows(data, window):
    X, y = [], []
    for i in range(len(data) - window):
        X.append(data[i : i + window])
        y.append(data[i + window])
    return np.stack(X).astype('float32'), np.stack(y).astype('float32')

X_all, y_all = make_windows(features_scaled, WINDOW)
print(f'Windows X : {X_all.shape}  (samples, window, features)')
print(f'Targets y : {y_all.shape}')

# 4. Train / Val / Test split (70 / 15 / 15)
n       = len(X_all)
n_train = int(0.70 * n)
n_val   = int(0.15 * n)

X_train = tf.constant(X_all[:n_train])
y_train = tf.constant(y_all[:n_train])
X_val   = tf.constant(X_all[n_train : n_train + n_val])
y_val   = tf.constant(y_all[n_train : n_train + n_val])
X_test  = tf.constant(X_all[n_train + n_val :])
y_test  = tf.constant(y_all[n_train + n_val :])

print(f'Train : {X_train.shape[0]:>6,} samples')
print(f'Val   : {X_val.shape[0]:>6,} samples')
print(f'Test  : {X_test.shape[0]:>6,} samples')


In [ ]:
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(features_scaled[:, 0], linewidth=0.5, color='steelblue', label='ind_1 (normalised)')
ax.axvline(n_train + WINDOW,         color='orange', linestyle='--', label='train/val split')
ax.axvline(n_train + n_val + WINDOW, color='red',    linestyle='--', label='val/test split')
ax.set_title('QuitoBench — selected series (ind_1, normalised)', fontsize=13)
ax.set_xlabel('Time step (minutes)')
ax.set_ylabel('Normalised value')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 3. Model Architectures

### 3.1  Plain GRU (Baseline)

```
Input  (B, T=24, 5)
  -> GRU(64)              (B, T, 64)   temporal modelling step 1
  -> GRU(32)              (B, T, 32)   temporal modelling step 2
  -> Dense(5, linear)     (B, T,  5)
  -> last time step       (B,  5)      one-step-ahead prediction
```

### 3.2  CNN-GRU  (Sajjad et al., IEEE Access 2020)

```
Input  (B, T=24, 5)
  -> Conv1D(16 filters, k=2, ReLU, SAME)  (B, T, 16)  local feature extraction
  -> Dropout(0.1)
  -> Conv1D( 8 filters, k=2, ReLU, SAME)  (B, T,  8)  feature compression
  -> Dropout(0.1)
  -> GRU(64)                              (B, T, 64)  temporal modelling step 1
  -> GRU(32)                              (B, T, 32)  temporal modelling step 2
  -> Dense(5, linear)                     (B, T,  5)
  -> last time step                       (B,  5)     one-step-ahead prediction
```

Both networks inherit from `TimeSeriesNet -> DeepNetwork`, reusing early stopping,
LR decay, gradient clipping, and checkpoint utilities from Project 4.


In [ ]:
INPUT_SHAPE = (WINDOW, N_FEATURES)   # (24, 5)

print('=' * 55)
print('GRU (Baseline)')
print('=' * 55)
gru_net = GRU_Regressor(INPUT_SHAPE, N_FEATURES, gru_units=(64, 32))
gru_net.compile(loss='mse', lr=1e-3, print_summary=True)

print()
print('=' * 55)
print('CNN-GRU (Sajjad et al., 2020)')
print('=' * 55)
cnn_gru = CNN_GRU_Regressor(
    INPUT_SHAPE, N_FEATURES,
    cnn_filters=(16, 8), kernel_size=2,
    dropout_rates=(0.1, 0.1), gru_units=(64, 32),
)
cnn_gru.compile(loss='mse', lr=1e-3, print_summary=True)


---
## 4. Training

Both models are trained with:
- **Optimizer**: Adam (lr = 1e-3)
- **Loss**: MSE
- **Gradient clipping**: global norm <= 1.0
- **Early stopping**: patience = 8 validation epochs
- **LR decay**: x0.5 when val loss plateaus (patience = 5), max 5 decays
- `@tf.function` on `train_step` and `test_step` for efficient graph execution


In [ ]:
BATCH_SIZE = 64
MAX_EPOCHS = 50

print('Training Plain GRU ...')
gru_train_loss, gru_val_loss, gru_epochs = gru_net.fit(
    X_train, y_train,
    x_val=X_val, y_val=y_val,
    batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
    val_every=1, verbose=True,
    patience=8, lr_patience=5, lr_decay_factor=0.5, lr_max_decays=5,
)
print(f'GRU finished in {gru_epochs} epochs.')


In [ ]:
print('Training CNN-GRU ...')
cnn_train_loss, cnn_val_loss, cnn_epochs = cnn_gru.fit(
    X_train, y_train,
    x_val=X_val, y_val=y_val,
    batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
    val_every=1, verbose=True,
    patience=8, lr_patience=5, lr_decay_factor=0.5, lr_max_decays=5,
)
print(f'CNN-GRU finished in {cnn_epochs} epochs.')


---
## 5. Comparison

We evaluate both models on the held-out test set using three regression metrics
(matching Sajjad et al.):

| Metric | Formula |
|---|---|
| MSE  | mean( (y - y_hat)^2 ) |
| MAE  | mean( |y - y_hat| ) |
| RMSE | sqrt( MSE ) |


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(gru_train_loss,  label='GRU train',     color='royalblue',  linewidth=1.5)
axes[0].plot(cnn_train_loss,  label='CNN-GRU train',  color='darkorange', linewidth=1.5)
axes[0].set_title('Training Loss (MSE)', fontsize=12)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(gru_val_loss,  label='GRU val',     color='royalblue',  linewidth=1.5)
axes[1].plot(cnn_val_loss,  label='CNN-GRU val',  color='darkorange', linewidth=1.5)
axes[1].set_title('Validation Loss (MSE)', fontsize=12)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Training & Validation Loss Curves', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
gru_mse, gru_mae, gru_rmse, gru_preds = gru_net.evaluate_regression(X_test, y_test)
cnn_mse, cnn_mae, cnn_rmse, cnn_preds = cnn_gru.evaluate_regression(X_test, y_test)

results = pd.DataFrame({
    'Model': ['GRU (Baseline)', 'CNN-GRU'],
    'MSE':   [gru_mse, cnn_mse],
    'MAE':   [gru_mae, cnn_mae],
    'RMSE':  [gru_rmse, cnn_rmse],
})
print('Test-set performance:')
print(results.to_string(index=False, float_format='{:.6f}'.format))


In [ ]:
metrics  = ['MSE', 'MAE', 'RMSE']
gru_vals = [gru_mse, gru_mae, gru_rmse]
cnn_vals = [cnn_mse, cnn_mae, cnn_rmse]

x, width = np.arange(len(metrics)), 0.35
fig, ax  = plt.subplots(figsize=(8, 5))
bars_g   = ax.bar(x - width/2, gru_vals, width, label='GRU (Baseline)', color='royalblue',  alpha=0.85)
bars_c   = ax.bar(x + width/2, cnn_vals, width, label='CNN-GRU',        color='darkorange', alpha=0.85)

top = max(gru_vals + cnn_vals)
for bar in list(bars_g) + list(bars_c):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + top*0.01,
            f'{bar.get_height():.5f}', ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x); ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylabel('Error (normalised scale)', fontsize=11)
ax.set_title('CNN-GRU vs Plain GRU — Test-Set Metrics', fontsize=13)
ax.legend(fontsize=11); ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
N_SHOW   = min(300, len(gru_preds))
y_actual = y_test.numpy()[:N_SHOW, 0]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(y_actual,              label='Actual',   color='steelblue',  alpha=0.85, linewidth=1.2)
axes[0].plot(cnn_preds[:N_SHOW, 0], label='CNN-GRU',  color='darkorange', alpha=0.85, linewidth=1.2)
axes[0].set_title('CNN-GRU — Actual vs Predicted (ind_1)', fontsize=12)
axes[0].set_ylabel('Normalised value'); axes[0].legend(fontsize=10); axes[0].grid(True, alpha=0.3)

axes[1].plot(y_actual,              label='Actual',         color='steelblue',   alpha=0.85, linewidth=1.2)
axes[1].plot(gru_preds[:N_SHOW, 0], label='GRU (Baseline)', color='forestgreen', alpha=0.85, linewidth=1.2)
axes[1].set_title('Plain GRU — Actual vs Predicted (ind_1)', fontsize=12)
axes[1].set_xlabel('Time step (minutes from test start)')
axes[1].set_ylabel('Normalised value'); axes[1].legend(fontsize=10); axes[1].grid(True, alpha=0.3)

plt.suptitle('QuitoBench Test Set: One-Step-Ahead Prediction', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
print('Metric   GRU          CNN-GRU      Delta (CNN-GRU improvement)')
print('-' * 65)
for metric, gv, cv in zip(metrics, gru_vals, cnn_vals):
    delta = (gv - cv) / gv * 100
    sign  = '+' if delta > 0 else ''
    print(f'{metric:<6}   {gv:.6f}     {cv:.6f}     {sign}{delta:.1f}%')
